# Pre-trained Models aur Transfer Learning

CNNs ko train karna kaafi time le sakta hai, aur iske liye bahut saara data chahiye hota hai. Lekin, zyadatar time network ke dwara images se patterns nikaalne ke liye use hone wale best low-level filters seekhne mein kharch hota hai. Ek natural sawaal uthta hai - kya hum ek dataset par trained neural network ko use karke, poori training process ke bina, usse alag-alag images ko classify karne ke liye adapt kar sakte hain?

Is approach ko **Transfer Learning** kehte hain, kyunki hum ek neural network model se doosre mein kuch knowledge transfer karte hain. Transfer Learning mein, hum aam taur par ek pre-trained model se shuru karte hain, jo kisi bade image dataset, jaise **ImageNet**, par trained hota hai. Yeh models pehle se hi general images se alag-alag features nikaalne mein achha kaam kar sakte hain, aur kai cases mein un nikaale gaye features ke upar ek classifier banana achhe results de sakta hai.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchinfo import summary
import numpy as np
import os

from pytorchcv import train, plot_results, display_dataset, train_long, check_image_dir

## Cats vs. Dogs Dataset

Is unit mein, hum cats aur dogs ki images ko classify karne ki ek real-life problem solve karenge. Iske liye hum [Kaggle Cats vs. Dogs Dataset](https://www.kaggle.com/c/dogs-vs-cats) use karenge, jo [Microsoft se bhi download kiya ja sakta hai](https://www.microsoft.com/en-us/download/details.aspx?id=54765).

Chalo is dataset ko download karte hain aur `data` directory mein extract karte hain (isme thoda time lag sakta hai!):

In [ ]:
if not os.path.exists('data/kagglecatsanddogs_5340.zip'):
    !wget -P data https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip

In [ ]:
import zipfile
if not os.path.exists('data/PetImages'):
    with zipfile.ZipFile('data/kagglecatsanddogs_5340.zip', 'r') as zip_ref:
        zip_ref.extractall('data')

Afsos ki baat hai, dataset mein kuch corrupted image files hain. Humein corrupted files check karne ke liye ek quick cleanup karna hoga. Is tutorial ko disturb na karne ke liye, humne dataset verify karne wala code ek module mein shift kar diya hai.

In [ ]:
check_image_dir('data/PetImages/Cat/*.jpg')
check_image_dir('data/PetImages/Dog/*.jpg')

Aage, chalo images ko PyTorch datasets mein load karte hain, unhein tensors mein convert karte hain aur kuch normalization karte hain. Hum `std_normalize` transform apply karenge taaki images pre-trained VGG network ke expected range mein aa jaayein:

In [ ]:
std_normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
trans = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(), 
        std_normalize])
dataset = torchvision.datasets.ImageFolder('data/PetImages',transform=trans)
trainset, testset = torch.utils.data.random_split(dataset,[20000,len(dataset)-20000])

display_dataset(dataset)

## Pre-trained Models

`torchvision` module ke andar kai alag-alag pre-trained models available hain, aur internet par aur bhi zyada models mil sakte hain. Chalo dekhte hain ki sabse simple VGG-16 model ko kaise load aur use kiya ja sakta hai:

In [ ]:
vgg = torchvision.models.vgg16(pretrained=True)
sample_image = dataset[0][0].unsqueeze(0)
res = vgg(sample_image)
print(res[0].argmax())

Jo result humein mila hai woh `ImageNet` class ka ek number hai, jise [yahan](https://gist.github.com/yrevar/942d3a0ac09ec9e5eb3a) dekha ja sakta hai. Hum neeche diye gaye code se is class table ko automatically load karke result nikaal sakte hain:

In [ ]:
import json, requests
class_map = json.loads(requests.get("https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json").text)
class_map = { int(k) : v for k,v in class_map.items() }

class_map[res[0].argmax().item()]

Chalo VGG-16 network ka structure bhi dekhte hain:

In [ ]:
summary(vgg,input_size=(1,3,224,224))

Jis layer ke baare mein hum pehle se jaante hain, uske alawa ek aur type ki layer hoti hai jise **Dropout** kehte hain. Yeh layers **regularization** technique ki tarah kaam karti hain. Regularization learning process mein halke changes karta hai taaki model better tarike se generalize kar sake. Training ke dauran, dropout layers pichli layer ke kuch neurons (lagbhag 30%) ko hata deti hain, aur unke bina training hoti hai. Yeh process optimization ko local minima se bahar nikaalne mein madad karti hai aur decision-making power ko alag-alag neural paths ke beech distribute karti hai, jisse network ki overall stability improve hoti hai.

## GPU par computations

VGG-16 jaise deep neural networks aur doosre modern architectures ko run karne ke liye kaafi zyada computational power chahiye hoti hai. Agar GPU available ho, toh GPU acceleration use karna samajhdaari hai. Aisa karne ke liye, humein computation mein shamil saare tensors ko explicitly GPU par transfer karna padta hai.

Aam taur par ise karne ka tarika yeh hai ki code mein GPU ki availability check ki jaaye, aur `device` naam ka ek variable define kiya jaaye, jo computational device - GPU ya CPU - ki taraf point karta hai.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print('Doing computations on device = {}'.format(device))

vgg.to(device)
sample_image = sample_image.to(device)

vgg(sample_image).argmax()

## VGG Features nikaalna

Agar hum apni images se features nikaalne ke liye VGG-16 use karna chahte hain, toh humein model ki final classification layers ke bina isse chahiye hoga. Asal mein, is "feature extractor" ko `vgg.features` method use karke paya ja sakta hai:

In [ ]:
res = vgg.features(sample_image).cpu()
plt.figure(figsize=(15,3))
plt.imshow(res.detach().view(512,-1).T)
print(res.size())

Feature tensor ka dimension 512x7x7 hai, lekin ise visualize karne ke liye humein ise 2D form mein badalna pada.

Ab chalo dekhte hain ki kya in features ko images classify karne ke liye use kiya ja sakta hai. Chalo manually kuch images ka portion lete hain (hamare case mein 800), aur unke feature vectors ko pehle se pre-compute kar lete hain. Hum result ko ek bade tensor mein store karenge jise `feature_tensor` kehte hain, aur labels ko `label_tensor` mein store karenge:

In [ ]:
bs = 8
dl = torch.utils.data.DataLoader(dataset,batch_size=bs,shuffle=True)
num = bs*100
feature_tensor = torch.zeros(num,512*7*7).to(device)
label_tensor = torch.zeros(num).to(device)
i = 0
for x,l in dl:
    with torch.no_grad():
        f = vgg.features(x.to(device))
        feature_tensor[i:i+bs] = f.view(bs,-1)
        label_tensor[i:i+bs] = l
        i+=bs
        print('.',end='')
        if i>=num:
            break


Ab hum `vgg_dataset` define kar sakte hain jo is tensor se data leta hai, ise `random_split` function se training aur test sets mein split karta hai, aur nikaale gaye features ke upar ek chhota single-layer dense classification network train karta hai:

In [ ]:
vgg_dataset = torch.utils.data.TensorDataset(feature_tensor,label_tensor.to(torch.long))
train_ds, test_ds = torch.utils.data.random_split(vgg_dataset,[700,100])

train_loader = torch.utils.data.DataLoader(train_ds,batch_size=32)
test_loader = torch.utils.data.DataLoader(test_ds,batch_size=32)

net = torch.nn.Sequential(torch.nn.Linear(512*7*7,2),torch.nn.LogSoftmax()).to(device)

history = train(net,train_loader,test_loader)

Result shaandar hai, hum lagbhag 98% probability ke saath cat aur dog mein difference bata sakte hain! Lekin, humne is approach ko sirf saari images ke ek chhote subset par hi test kiya hai, kyunki manual feature extraction mein kaafi time lagta hai.

## VGG network use karke Transfer Learning

Hum features ko manually pehle se pre-compute karne se bhi bach sakte hain, aur iske bajaye training ke dauran original VGG-16 network ko poori tarah use kar sakte hain. Chalo VGG-16 object ke structure par nazar daalte hain:

In [ ]:
print(vgg)

Aap dekh sakte hain ki network mein yeh shamil hain:
* Feature extractor (`features`), jisme kai convolutional aur pooling layers shamil hain
* Average pooling layer (`avgpool`)
* Final `classifier`, jisme kai dense layers hain, jo 25088 input features ko 1000 classes (jo ImageNet mein classes ki sankhya hai) mein badalti hain

Hamare dataset ko classify karne ke liye end-to-end model train karne ke liye, humein yeh karna hoga:
* **Final classifier ko replace karein** taaki yeh required number of classes produce kar sake. Hamare case mein, hum 25088 inputs aur 2 output neurons wali ek `Linear` layer use kar sakte hain.
* **Convolutional feature extractor ke weights ko freeze karein**, taaki woh train na ho. Shuru mein aisi freezing recommended hai, kyunki warna untrained classifier layer original pre-trained convolutional extractor ke weights ko kharab kar sakti hai. Weights freeze karna sabhi parameters ki `requires_grad` property ko `False` set karke kiya ja sakta hai.

In [ ]:
vgg.classifier = torch.nn.Linear(25088,2).to(device)

for x in vgg.features.parameters():
    x.requires_grad = False

summary(vgg,(1, 3,244,244))

Jaisa ki aap summary se dekh sakte hain, is model mein lagbhag 15 million total parameters hain, lekin inmein se sirf 50,000 hi trainable hain - yeh classification layer ke weights hain. Yeh achhi baat hai, kyunki hum kam examples ke saath kam parameters ko fine-tune kar sakte hain.

Ab chalo hamare original dataset ka use karke model ko train karte hain. Yeh process kaafi time legi, isliye hum `train_long` function use karenge, jo poore epoch ke khatam hone ka intezaar kiye bina kuch intermediate results print karega. Yeh strongly recommend kiya jaata hai ki is training ko GPU-enabled compute par run kiya jaaye!

In [ ]:
trainset, testset = torch.utils.data.random_split(dataset,[20000,len(dataset)-20000])
train_loader = torch.utils.data.DataLoader(trainset,batch_size=16)
test_loader = torch.utils.data.DataLoader(testset,batch_size=16)

train_long(vgg,train_loader,test_loader,loss_fn=torch.nn.CrossEntropyLoss(),epochs=1,print_freq=90)

In [ ]:
torch.save(vgg,'data/cats_dogs.pth')

Hum phir kisi bhi samay file se model load kar sakte hain. Aap ise useful paayenge agar agla experiment model ko kharab kar de - aapko shuru se phir se start nahi karna padega.

In [ ]:
vgg = torch.load('data/cats_dogs.pth')

## Transfer Learning ki Fine-tuning

Pichhle section mein, humne final classifier layer ko apne dataset ki images classify karne ke liye train kiya tha. Lekin, humne feature extractor ko dobara train nahi kiya tha, aur hamara model un features par depend kar raha tha jo model ne ImageNet data par seekhe the. Agar aapke objects general ImageNet images se visually alag hain, toh features ka yeh combination sabse achha kaam nahi kar sakta. Isliye, yeh samajhdaari hogi ki hum convolutional layers ko bhi train karna shuru karein.

Iske liye, hum un convolutional filter parameters ko unfreeze kar sakte hain jinhein humne pehle freeze kiya tha.

> **Note:** Yeh important hai ki aap pehle parameters ko freeze karein aur classification layer ke weights ko stabilize karne ke liye kuch epochs tak training karein. Agar aap turant unfrozen parameters ke saath end-to-end network ki training shuru karte hain, toh bade errors pre-trained weights ko convolutional layers mein destroy kar sakte hain.

In [ ]:
for x in vgg.features.parameters():
    x.requires_grad = True

Unfreeze karne ke baad, hum kuch aur epochs tak train kar sakte hain. Aap kam learning rate bhi chun sakte hain, taaki pre-trained weights par asar kam ho. Lekin, kam learning rate ke bawajood, aap umeed kar sakte hain ki training ki shuruaat mein accuracy kam ho jaayegi, lekin aakhir mein yeh fixed weights wale case se thode zyada level tak pahunch jaayegi.

> **Note:** Yeh training bahut dheemi gati se hoti hai, kyunki humein network ki kai layers se gradients ko wapas propagate karna padta hai! Aap shuruaati kuch minibatches dekhkar trend ka andaaza laga sakte hain, aur phir computation ko rok sakte hain.

In [ ]:
train_long(vgg,train_loader,test_loader,loss_fn=torch.nn.CrossEntropyLoss(),epochs=1,print_freq=90,lr=0.0001)

## Doosre Computer Vision Models

VGG-16 sabse simple computer vision architectures mein se ek hai. `torchvision` package kai aur pre-trained networks provide karta hai. Inmein se sabse zyada use hone wale networks hain **ResNet** architectures, jo Microsoft ne develop kiye hain, aur Google ka **Inception**. Jaise, chalo sabse simple ResNet-18 model ke architecture ko dekhte hain (ResNet models ka ek family hai jisme alag-alag depth hoti hai, aap ResNet-151 ke saath experiment kar sakte hain agar aap dekhna chahte hain ki ek sach mein deep model kaisa dikhta hai):

In [ ]:
resnet = torchvision.models.resnet18()
print(resnet)

Jaisa ki aap dekh sakte hain, model mein wahi building blocks shamil hain: feature extractor aur final classifier (`fc`). Isse hum is model ko bilkul usi tarah use kar sakte hain jaise humne transfer learning ke liye VGG-16 use kiya tha. Aap upar diye gaye code ke saath experiment kar sakte hain, alag-alag ResNet models ko base model ki tarah use karke dekhein ki accuracy kaise badalti hai.

## Batch Normalization

Is network mein ek aur type ki layer shamil hai: **Batch Normalization**. Batch Normalization ka idea yeh hai ki neural network se flow hone wali values ko sahi interval mein laaya jaaye. Aam taur par, neural networks tab sabse achha kaam karte hain jab saari values [-1,1] ya [0,1] ki range mein hoti hain, aur isi wajah se hum apne input data ko usi ke hisaab se scale/normalize karte hain. Lekin, ek deep network ki training ke dauran aisa ho sakta hai ki values is range se kaafi bahar chali jaayein, jisse training mein problem ho sakti hai. Batch Normalization layer current minibatch ki saari values ke liye average aur standard deviation calculate karti hai aur unhein normalize karne ke liye use karti hai, isse pehle ki signal ko neural network layer se pass kiya jaaye. Yeh deep networks ki stability ko kaafi had tak improve karta hai.

## Key Takeaways

Transfer Learning use karke, hum apne custom object classification task ke liye jaldi se ek classifier taiyaar kar paaye aur high accuracy hasil ki. Lekin, yeh example poori tarah fair nahi tha, kyunki original VGG-16 network ko pehle se hi cats aur dogs pehchaanne ke liye train kiya gaya tha, aur isliye hum network mein pehle se maujood zyadatar patterns ko hi dobara use kar rahe the. Aap umeed kar sakte hain ki zyada specialized domain-specific objects, jaise kisi plant ki production line par details ya alag-alag trees ke patton par, accuracy kam ho sakti hai.

Aap dekh sakte hain ki ab hum jo zyada complex tasks solve kar rahe hain, unhein high computational power chahiye aur unhein aasaani se CPU par solve nahi kiya ja sakta. Agle unit mein, hum kam compute resources use karke wahi model train karne ke liye ek lightweight implementation use karne ki koshish karenge, jisse accuracy mein sirf maamooli kami aayegi.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->